In [39]:
"""
importing libraries
"""
import json
import numpy as np
import pickle as pl
from datasets import load_dataset
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch
import pandas as pd
import seaborn as sns

In [ ]:
output_dir = '/home/aditya/hack_model/snli_(0.01, 0.44, 0.55)'

In [ ]:
## my input features
alignment_matrix = np.load(f"{output_dir}/alignment_matrix_M.npy")

In [ ]:
with open(f'{output_dir}/accuracy_arr.pkl','rb') as f:
    accuracy_arr = pl.load(f)

In [ ]:
with open(f'{output_dir}/dataset_info.json') as f:
    dataset_info =json.load(f)

In [ ]:
labels = dataset_info['dataset']

In [2]:
kfold_models_dir = "./kfold_models"
kfold_results_dir = "./kfold_results"
kfold_models_visualize_dir = "./kfold_models_visualize"
datainfo_dir = "./results_datainfo"
training_data_dir = "./npy_files"
align_matrices_dir = "./results_align_matrix"
datainfo_dir = "./results_datainfo"

#### normalizing and saving alignment scores

In [ ]:
def normalize_max(matrix):
    """
    Scale to [0,1] per expert
    """
    max_val = matrix.max(axis=0, keepdims=True)
    max_val = np.where(max_val == 0, 1, max_val)  # Avoid division by zero
    return matrix / max_val

def normalize_standard(matrix):
    """
    Z-score per expert
    """
    mean = matrix.mean(axis=0, keepdims=True)
    std = matrix.std(axis=0, keepdims=True, ddof=0)
    return (matrix - mean) / np.where(std == 0, 1, std)

def normalize_all(dataset, interpolation, base_dir, norm_type):
    """
    Normalize all matrices for dataset/interpolation
    """
    norm_fn = normalize_max if norm_type == 'max' else normalize_standard
    base_path = Path(base_dir)
    
    # Find all experiment directories
    dirs = [d for d in base_path.iterdir() 
            if d.is_dir() and d.name.startswith(f"{dataset}_{interpolation}_[")]
    
    stats = {'success': 0, 'skipped': 0}
    
    for exp_dir in tqdm(dirs, desc=f"{dataset}/{interpolation}/{norm_type}"):
        input_file = exp_dir / f"alignment_matrix_{interpolation}.npy"
        output_file = exp_dir / f"alignment_matrix_{interpolation}_{norm_type}.npy"
        
        # Load, normalize, save
        matrix = np.load(input_file)
        normalized = norm_fn(matrix)
        np.save(output_file, normalized)
        stats['success'] += 1
    
    print(f"  {dataset}/{interpolation}/{norm_type}: ✓{stats['success']} ⊘{stats['skipped']}")
    return stats


if __name__ == "__main__":
    
    DATASETS = ['ag_news', 'yelp_review']
    INTERPOLATIONS = ['linear', 'slerp', 'ties', 'model_baseline']
    NORMALIZATIONS = ['max',"standard"]
    models = ["bert","gpt2"]
    
    for model_name in models:
        for dataset in DATASETS:
            base_dir = align_matrices_dir + f"/{model_name}/{dataset}"
            for interp in INTERPOLATIONS:
                for norm in NORMALIZATIONS:
                    stats = normalize_all(dataset, interp, base_dir, norm)
                    print(stats)
    
    print("\n✅ Done!")

### Validating the alignment scores

In [ ]:
align_dir = Path("./results_align_matrix")
iter = align_dir.iterdir()
# [print(dir) for dir in align_dir.iterdir()]


for dir in iter:
    file_iter = dir.iterdir()
    for file in file_iter:
        file_name = str(file)
        if("alignment_matrix" in file_name ):
            alignment_matrix = np.load(file_name)
            print(max(alignment_matrix[:,0]))

#### Getting training points

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold  # ← Just for splitting indices!
import os
import cvxpy as cp
from cvxpylayers.torch import CvxpyLayer

In [5]:
import json
import numpy as np
from pathlib import Path
import re

def parse_proportion_from_filename(filename):
    """
    Extract proportion array from filename like 'ag_news_6e-06_[0.1, 0.3, 0.3, 0.3].json'
    """
    match = re.search(r'[\[\(]\s*([0-9eE+\-.,\s]+)\s*[\]\)]', filename)
    if match:
        arr = [float(x) for x in match.group(1).split(",") if x.strip()]
        return arr
    return None

def load_experiment_data(dataset_name, interpolation_name, normalization, base_dir, data_dir):
    """
    Load experiment results with their proportions.

    Args:
        dataset_name: Dataset identifier ('snli' or 'ag_news')
        interpolation_name: Method name ('linear', 'slerp', 'ties', 'model_baseline')
        base_dir: Directory containing alignment result folders
        data_dir: Directory containing dataset folders with dataset_info.json

    Returns:
        list[dict]: Loaded experiment records
    """
    base_path = Path(base_dir)
    data_path = Path(data_dir)

    print(f"Loading {dataset_name} dataset from HuggingFace...")
    
    if(dataset_name!="yelp_review"):
        hf_dataset = _load_hf_train_split(dataset_name)
    else:
        hf_dataset = _load_hf_train_split('yelp/yelp_review_full') 
    print(f"Dataset loaded: {len(hf_dataset)} samples")

    experiment_dirs = sorted(
        d for d in base_path.iterdir()
        if d.is_dir() and d.name.startswith(f'{dataset_name}_{interpolation_name}_[')
    )

    print(f"Scanning {len(experiment_dirs)} directories...")

    experiments = []

    for exp_dir in experiment_dirs:
        proportions = parse_proportion_from_filename(exp_dir.name)
        if proportions is None:
            print(f"  ⚠ Could not parse proportions: {exp_dir.name}")
            continue

        data_config_name = f"{dataset_name}_({', '.join(map(str, proportions))})"
        dataset_info_file = data_path / data_config_name / "dataset_info.json"
        alignment_file = exp_dir / f"alignment_matrix_{interpolation_name}_{normalization}.npy"

        if not dataset_info_file.exists():
            print(f"  ⚠ Missing dataset_info.json: {dataset_info_file}")
            continue

        if not alignment_file.exists():
            print(f"  ⚠ Missing alignment matrix: {alignment_file}")
            continue

        with dataset_info_file.open() as f:
            dataset_info = json.load(f)

        indices_D = dataset_info["indices_D"]
        labels = [hf_dataset[idx]["label"] for idx in indices_D]
        alignment_matrix = np.load(alignment_file)

        experiments.append({
            "proportions": proportions,
            "alignment_matrix": alignment_matrix,
            "labels": labels,
            "output_dir": str(exp_dir),
            "config_name": exp_dir.name,
            "interpolation": interpolation_name,
            "normalization": normalization
        })

    print(f"\n✓ Loaded {len(experiments)} experiments for {interpolation_name}")

    experiments.sort(key=lambda x: tuple(x["proportions"]))
    return experiments


def _load_hf_train_split(dataset_name):
    return load_dataset(dataset_name, split="train")

def compute_mean_alignment_per_expert_per_class(alignment_matrix, class_labels):
    """
    Compute mean alignment score for each pseudo-expert for each class.
    
    Args:
        alignment_matrix: np.ndarray, shape (n_samples, n_pseudo_experts)
        class_labels: 1D array-like of length n_samples with class labels
    
    Returns:
        dict: {class_id: np.array([mean_expert_0, mean_expert_1, ...])}
    """
    class_labels = np.array(class_labels)
    unique_classes = np.unique(class_labels)
    n_pseudo_experts = alignment_matrix.shape[1]
    
    results = {}
    
    for cls in unique_classes:
        cls_mask = (class_labels == cls)
        cls_alignment = alignment_matrix[cls_mask]  # (n_samples_cls, n_experts)
        
        if cls_alignment.shape[0] == 0:
            # No samples for this class
            continue
        
        # Mean across samples for each expert
        mean_per_expert = cls_alignment.mean(axis=0)  # shape (n_experts,)
        results[cls] = mean_per_expert
    return results


def compute_std_dev_alignment_per_expert_per_class(alignment_matrix, class_labels):
    """
    Compute std dev of alignment score for each pseudo-expert for each class.
    
    Args:
        alignment_matrix: np.ndarray, shape (n_samples, n_pseudo_experts)
        class_labels: 1D array-like of length n_samples with class labels
    
    Returns:
        dict: {class_id: np.array([std_expert_0, std_expert_1, ...])}
    """
    class_labels = np.array(class_labels)
    unique_classes = np.unique(class_labels)
    n_pseudo_experts = alignment_matrix.shape[1]
    
    results = {}

    for cls in unique_classes:
        cls_mask = (class_labels == cls)
        cls_alignment = alignment_matrix[cls_mask]  # (n_samples_cls, n_experts)
        
        if cls_alignment.shape[0] == 0:
            # No samples for this class
            continue
        
        # Mean across samples for each expert
        std_per_expert = cls_alignment.std(axis=0, ddof=0)  
        results[cls] = std_per_expert 
    
    return results

In [9]:
def get_all_scores(dataset_name, interpolation_name, normalization, base_dir,data_dir):
    """
    Load experiments and compute mean/std alignment per class for each.

    Args:
        dataset_name (str): Dataset identifier.
        interpolation_name (str): Interpolation method name.
        normalization (str): Normalization setting.
        base_dir (str | Path): Base directory for experiments.
        data_dir (str | Path): Data directory.

    Returns:
        tuple: (proportions_list, mean_per_class_list, std_per_class_list)
    """
    # Load all experiments using existing function
    experiments = load_experiment_data(dataset_name, interpolation_name, normalization,  base_dir,data_dir)
    print(f"Loaded {len(experiments)} experiments for {interpolation_name}")
    
    proportions_arr = []
    mean_per_class_arr = []
    std_per_class_arr = []
    
    for exp in experiments:
        
        proportion = exp['proportions']
        alignment_matrix = exp['alignment_matrix']
        labels = exp['labels']
        
        # Use existing function to compute mean per class
        mean_per_class = compute_mean_alignment_per_expert_per_class(
            alignment_matrix, labels
        )
        
        std_per_class = compute_std_dev_alignment_per_expert_per_class(
            alignment_matrix, labels)
        
        proportions_arr.append(proportion)
        mean_per_class_arr.append(mean_per_class)
        std_per_class_arr.append(std_per_class)
        
    return proportions_arr, mean_per_class_arr, std_per_class_arr

In [10]:
def extract_training_data_for_simple(proportions_arr,mean_per_class_arr,std_dev_per_class_arr, n_classes,n_pseudoexperts,stdOrNot):
    """
    Extract training data for FCN using existing functions.
    
    Returns:
        X: np.array of shape (n_experiments, n_features)
           Features = mean alignment scores per class per expert (3 * 15 = 45)
        y: np.array of shape (n_experiments, n_classes)
           Target class proportions
        metadata: list of dicts with experiment info
    """
    
    X_list = []
    y_list = []
    
    for i in range(len(proportions_arr)):
        
        mean_per_class = mean_per_class_arr[i]
        std_dev_per_class = std_dev_per_class_arr[i]
        
        # Build feature vector: concatenate mean alignment for all classes
        feature_vector = []
        for cls in sorted(mean_per_class.keys()):  
            feature_vector.extend(mean_per_class[cls])  # 15 values per class   
            if(stdOrNot):
                feature_vector.extend(std_dev_per_class[cls])  # 15 values per class
            
        X_list.append(feature_vector)
        y_list.append(proportions_arr[i])
    
    X = np.array(X_list)
    y = np.array(y_list)
    
    print(f"\n✓ Extracted {len(X)} training samples")
    print(f"  X shape: {X.shape}")
    print(f"  y shape: {y.shape}")
    
    return X, y

In [11]:
from itertools import combinations

def extract_training_data_for_optimizer(proportions_arr, mean_per_class_arr, n_classes=3):
    """
    Extract pairwise training data for optimizer.
    
    Returns:
        X: (n_samples, 30) - features for 2 classes
        y: (n_samples, 2) - proportions for 2 classes
    """
    
    X_list = []
    y_list = []
    metadata_list = []
    
    # All possible pairs: (0,1), (0,2), (1,2) # least class number first 
    class_pairs = list(combinations(range(n_classes), 2))
    
    for proportions, mean_per_class in zip(proportions_arr, mean_per_class_arr):
        
        # For each pair, create one sample
        for class_i, class_j in class_pairs:
            
            if class_i not in mean_per_class or class_j not in mean_per_class:
                continue
            
            # Features: concat alignments from both classes
            features = np.concatenate([
                mean_per_class[class_i],  # 15 values
                mean_per_class[class_j]   # 15 values
            ])  # Total: 30
            
            # Targets: proportions for this pair
            targets = np.array([proportions[class_i], proportions[class_j]])
            
            class_indices = np.array([class_i, class_j])
            X_list.append(features)
            y_list.append(targets)
            metadata_list.append(class_indices)

    
    X = np.array(X_list)
    y = np.array(y_list)
    metadata = np.array(metadata_list)
    
    print(f"✓ Extracted {len(X)} samples")
    print(f"  X shape: {X.shape}")
    print(f"  y shape: {y.shape}")
    
    return X, y, metadata

In [ ]:
# Usage
if __name__ == "__main__":
    datasets = ['ag_news','yelp_review']
    interpolations = ['model_baseline', 'slerp', 'ties', 'linear']
    normalizations = ['max']
    n_pseudoexperts = 15
    statistics = ["mean"]
    models = ["gpt2"]
    
    for model_name in models:
        for statistic in statistics:
            for dataset in datasets:
                for interpolation in interpolations:
                    for normalization in normalizations:

                        base_dir = align_matrices_dir + f"/{model_name}/{dataset}"
                        data_dir = datainfo_dir + f"/{model_name}/{dataset}"
                    
                        proportions_arr, mean_per_class_arr, std_dev_per_class_arr = get_all_scores(dataset, interpolation, normalization, base_dir,data_dir)
                        
                        n_classes = len(proportions_arr[0])        
                        
                        if(statistic=="mean"):
                            X_simple, y_simple = extract_training_data_for_simple(proportions_arr, mean_per_class_arr, std_dev_per_class_arr, n_classes, n_pseudoexperts,False)
                            
                        elif (statistic=="mean&std"):
                            X_simple, y_simple = extract_training_data_for_simple(proportions_arr, mean_per_class_arr, std_dev_per_class_arr, n_classes, n_pseudoexperts,True)


                        output_path = Path(training_data_dir + f"/{model_name}/{dataset}")
                        output_path.mkdir(parents=True, exist_ok=True)

                        np.save(f'{output_path}/X_{dataset}_{interpolation}_{normalization}_simple_{statistic}.npy', X_simple)
                        np.save(f'{output_path}/y_{dataset}_{interpolation}_{normalization}_simple_{statistic}.npy', y_simple)
                        
                        print(f"✓ Saved {interpolation}\n")

#### Kfold script

In [ ]:
class ProportionFCN(nn.Module):
    """
    FCN to predict class proportions from alignment metrics.
    
    Input: (batch, 45) - mean alignment scores [class0_e0...class0_e14, class1_e0...class1_e14, class2_e0...class2_e14]
    Output: (batch, 3) - class proportions that sum to 1
    """
    def __init__(self, n_classes, no_of_stats, n_experts, hidden_dim):
        super().__init__()
        
        input_dim = n_classes * n_experts * no_of_stats
        
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            # nn.Linear(hidden_dim, hidden_dim),
            # nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_classes)
            # nn.Linear(input_dim, n_classes)
        )
    
    def forward(self, x):
        logits = self.network(x)
        proportions = F.softmax(logits, dim=1)
        return proportions

In [41]:
# eariler learnign rate: 1e-3
# different learnign rate for both datasets for a particular model and different learnign rate from bert itself
# we can have different configs for the fcn 

In [44]:
def train_fcn_with_kfold(X, y, model_class, n_classes, n_stats , no_of_layers,model_name, n_folds=10, epochs=100, batch_size=8, lr=1e-3, device='cuda'):
    """
    K-Fold CV with PyTorch model - FIXED VERSION
    
    Args:
        X: Input features (numpy array)
        y: Target labels (numpy array)
        model_class: Callable that returns a fresh model instance
        n_folds: Number of folds for cross-validation
        epochs: Number of training epochs per fold
        batch_size: Batch size for training
        lr: Learning rate
        device: Device to train on ('cuda' or 'cpu')
    
    Returns:
        fold_results: List of dicts with metrics for each fold
    """


    # ============================================
    #  K fold training pipeline
    # ============================================
    X_tensor = torch.FloatTensor(X)
    y_tensor = torch.FloatTensor(y)
    
    kfold = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_results = []
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(X)):
        
        # ============================================
        #  Training pipeline 
        # ============================================
        print(f"\n{'='*60}")
        print(f"Fold {fold + 1}/{n_folds}")
        print(f"{'='*60}")
        
        # ✅ FIX 1: Create fresh model for each fold
        model = model_class(n_classes,n_stats).to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.L1Loss()
        
        X_train, X_val = X_tensor[train_idx], X_tensor[val_idx]
        y_train, y_val = y_tensor[train_idx], y_tensor[val_idx]
        
        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        
        best_val_loss = float('inf')
        best_epoch = 0
        
        for epoch in range(epochs):
            # Training phase
            model.train()
            train_loss = 0.0
            
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                
                optimizer.zero_grad()
                pred = model(batch_X)
                loss = criterion(pred, batch_y)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
            
            train_loss /= len(train_loader)
            
            # Validation phase
            model.eval()
            with torch.no_grad():
                val_pred = model(X_val.to(device))
                val_loss = criterion(val_pred, y_val.to(device)).item()
            
            # ✅ FIX 2: Only print at intervals
            if (epoch + 1) % 20 == 0:
                print(f"Epoch {epoch+1:3d}: Train={train_loss:.6f}, Val={val_loss:.6f}")



        # ============================================
        # Evaluate best model on both train and val sets
        # ============================================
        model.eval()
        
        with torch.no_grad():
            # Validation metrics
            val_pred = model(X_val.to(device))
            val_mse = nn.MSELoss()(val_pred, y_val.to(device)).item()
            val_mae = torch.mean(torch.abs(val_pred - y_val.to(device))).item()
            
            # Training metrics (on best model)
            train_pred = model(X_train.to(device))
            train_mse = nn.MSELoss()(train_pred, y_train.to(device)).item()
            train_mae = torch.mean(torch.abs(train_pred - y_train.to(device))).item()
        
        print(f"  Train MSE: {train_mse:.6f} | Train MAE: {train_mae:.6f}")
        print(f"  Val   MSE: {val_mse:.6f} | Val   MAE: {val_mae:.6f}")

        fold_results.append({
            'fold': fold + 1,
            'train_mse': train_mse,
            'train_mae': train_mae,
            'val_mse': val_mse,
            'val_mae': val_mae
        })
        
        # Show sample predictions at the end of fold
        print(f"\nSample Predictions (first 3):")
        for i in range(min(3, len(val_pred))):
            print(f"  Pred: {val_pred[i].cpu().numpy()} | True: {y_val[i].cpu().numpy()}")


    # ============================================
    # Saving the final model
    # ============================================
    model = model_class(n_classes,n_stats).to(device)
    train_dataset = TensorDataset(X_tensor, y_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            pred = model(batch_X)
            loss = criterion(pred, batch_y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # Validation phase
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val.to(device))
            val_loss = criterion(val_pred, y_val.to(device)).item()
        
        # ✅ FIX 2: Only print at intervals
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1:3d}: Train={train_loss:.6f}, Val={val_loss:.6f}")
    
    kfold_model_path = kfold_models_dir + f"/{model_name}/{dataset}/model_{interpolation}_{normalization}_simple_{no_of_layers}_{statistic}.pt"
    torch.save(model.state_dict(),kfold_model_path)
    
    
    
    # ============================================
    # Aggregate results across all folds
    # ============================================
    print(f"\n{'='*60}")
    print("Overall K-Fold CV Results:")
    print(f"{'='*60}")
    
    # Training metrics
    avg_train_mse = np.mean([r['train_mse'] for r in fold_results])
    std_train_mse = np.std([r['train_mse'] for r in fold_results])
    avg_train_mae = np.mean([r['train_mae'] for r in fold_results])
    std_train_mae = np.std([r['train_mae'] for r in fold_results])
    
    # Validation metrics
    avg_val_mse = np.mean([r['val_mse'] for r in fold_results])
    std_val_mse = np.std([r['val_mse'] for r in fold_results])
    avg_val_mae = np.mean([r['val_mae'] for r in fold_results])
    std_val_mae = np.std([r['val_mae'] for r in fold_results])
    
    print(f"\nTraining Set:")
    print(f"  Average MSE: {avg_train_mse:.6f} ± {std_train_mse:.6f}")
    print(f"  Average MAE: {avg_train_mae:.6f} ± {std_train_mae:.6f}")
    
    print(f"\nValidation Set:")
    print(f"  Average MSE: {avg_val_mse:.6f} ± {std_val_mse:.6f}")
    print(f"  Average MAE: {avg_val_mae:.6f} ± {std_val_mae:.6f}")
    
    fold_results.append({
        'avg_train_mse': avg_train_mse ,
        'std_train_mse': std_train_mse,
        'avg_train_mae': avg_train_mae,
        'std_train_mae': std_train_mae,
        'avg_val_mse' : avg_val_mse,
        'std_val_mse' : std_val_mse,
        'avg_val_mae' : avg_val_mae,
        'std_val_mae' : std_val_mae
    })
    
    return fold_results

In [ ]:
if __name__ == "__main__":
    device = 'cuda'
    
    def create_model(n_classes,no_of_stats):
        return ProportionFCN(n_classes, no_of_stats, n_experts = 15, hidden_dim = 64)
    
    results_simple = {}
    
    # datasets = ['ag_news','yelp_review']
    # interpolations = ['model_baseline', 'slerp', 'ties', 'linear']
    # normalizations = ['max','standard']
    # statistics = ['mean','mean&std']
    # no_of_layers = 2
    # models = ["bert"]
    
    datasets = ['ag_news','yelp_review']
    interpolations = ['model_baseline', 'slerp', 'ties', 'linear']
    normalizations = ['max']
    statistics = ['mean']
    no_of_layers = 4
    models = ["gpt2"]
    
    for model_name in models:
        for statistic in statistics:
            if(statistic=="mean"):
                n_stats = 1
            else:
                n_stats = 2
            for normalization in normalizations:
                for dataset in datasets:
                    for interpolation in interpolations:
                        print(f"\n{'='*80}")
                        print(f"Training on {interpolation.upper()}")
                        print(f"{'='*80}")
                        
                        X_path = training_data_dir + f"/{model_name}/{dataset}" f'/X_{dataset}_{interpolation}_{normalization}_simple_{statistic}.npy'
                        y_path = training_data_dir + f"/{model_name}/{dataset}" f'/y_{dataset}_{interpolation}_{normalization}_simple_{statistic}.npy'
                        X = np.load(X_path)
                        y = np.load(y_path)
                        print(f"Data: X={X.shape}, y={y.shape}")
                        
                        n_classes = y.shape[1]
                        
                        # ✅ Pass model factory, not instance
                        result = train_fcn_with_kfold(
                            X, y, 
                            create_model,
                            n_classes,
                            n_stats,
                            no_of_layers,
                            model_name,
                            n_folds=10, 
                            epochs=50, 
                            batch_size=8, 
                            lr=1e-3, 
                            device=device
                        )
            
                        print("\n✓ Training complete!")

                        # pkl_dir_path = kfold_results_dir + f"/{model_name}/{dataset}"
                        # pkl_dir_path_posix = Path(pkl_dir_path)
                        # pkl_dir_path_posix.mkdir(parents=True, exist_ok=True)
                        # pkl_path = pkl_dir_path + f"/results_{dataset}_{interpolation}_{normalization}_simple_{no_of_layers}_{statistic}.pkl"
                        # with open(pkl_path,'wb') as f:
                        #     pl.dump(result,f)

In [ ]:
# ratios by the fcn
# optimizer ratios 
# true ratios 

# loss equation = swuared erro between true ratios and optimzier ratios 
# loss equation could have been squared error between true ratios and ratios by fcn and then we can use optimizer during infernce

## Text to see overview of k fold results

In [ ]:
datasets = ['ag_news','yelp_review']
interpolations = ['model_baseline', 'slerp', 'ties', 'linear']
normalizations = ['max']
layers = [2,3]
statistics = ['mean']
models = ["gpt2"]

for model_name in models:
    for normalization in normalizations:
        for statistic in statistics:
            for dataset in datasets:
                for interpolation in interpolations:
                    for layer in layers:
                        pkl_dir_path = kfold_results_dir + f"/{model_name}/{dataset}"
                        pkl_dir_path_posix = Path(pkl_dir_path)
                        pkl_dir_path_posix.mkdir(parents=True, exist_ok=True)
                        pkl_path = pkl_dir_path + f"/results_{dataset}_{interpolation}_{normalization}_simple_{layer}_{statistic}.pkl"
                        with open(pkl_path,'rb') as f:
                            dic = pl.load(f)[-1]
                            avg_train_mae = round(dic['avg_train_mae'],3)
                            std_train_mae = round(dic['std_train_mae'],3)
                            avg_val_mae = round(dic['avg_val_mae'],3)
                            std_val_mae = round(dic['std_val_mae'],3)
                            print(f"{dataset}_{interpolation}_{normalization}")
                            print(f"{avg_train_mae}±{std_train_mae}")
                            print(f"{avg_val_mae}±{std_val_mae}")
                            print("\n")

In [ ]:
# we want the model to understand the pseudoexpert specific information 
# we want the relative difference in alignment score to be really highlighted 
# we want the sign and magnitude to be conserved 
# the relative information of the pseudoexperts 

## Visualisations to evaluate weight matrix heatmaps

In [ ]:
# no of classes * no of pseduoexperts 
# all pseudoepxerts for a class contugously

In [ ]:
datasets = ['ag_news','yelp_review']
interpolations = ['model_baseline', 'slerp', 'ties', 'linear']
normalizations = ['max']
layers = [1]
statistics = ['mean']
models = ["bert"]

In [ ]:
# from pathlib import Path
# import matplotlib.pyplot as plt
# import torch


def visualize_weight_matrix(model_state_path, output_path, no_of_layers=None):
    output_dir = Path(output_path)
    output_dir.mkdir(parents=True, exist_ok=True)

    state_dict = torch.load(model_state_path, map_location="cpu")

    # collect only 2D weight tensors (linear layers)
    layers = [
        (name, tensor)
        for name, tensor in state_dict.items()
        if isinstance(tensor, torch.Tensor) and tensor.ndim == 2
    ]

    if no_of_layers is not None:
        layers = layers[:no_of_layers]

    total_layers = len(layers)

    for idx, (name, tensor) in enumerate(layers):
        weight = tensor.detach().cpu().float().numpy()

        plt.figure(figsize=(8, 6))
        plt.imshow(weight[0].reshape(15,4), aspect="auto")
        plt.colorbar()

        plt.xlabel("Input dimension")

        # 🔥 SPECIAL: last layer → class labels
        if idx == total_layers - 1:
            num_classes = weight.shape[0]

            plt.ylabel("Classes")
            plt.yticks(
                ticks=range(num_classes),
                labels=[f"class {i}" for i in range(num_classes)]
            )
            plt.title(f"{name} (Output Layer: {num_classes} classes)")
        else:
            plt.ylabel("Output dimension")
            plt.title(name)

        plt.tight_layout()

        safe_name = name.replace(".", "_").replace("/", "_")
        plt.savefig(output_dir / f"{idx:02d}_{safe_name}.png", dpi=200)
        plt.close()

    print(f"Saved {total_layers} layer visualizations to {output_dir}")

In [ ]:
def visualize_weight_matrix(model_state_path, output_path, no_of_layers=None):
    """
    Visualize weight matrices with separate heatmaps for each output class.
    
    For a 1-layer network:
    - Input: (60,) = 4 classes × 15 experts (organized as [C0_E0...C0_E14, C1_E0...C1_E14, ...])
    - Output: (4,) = 4 classes
    - Creates 4 separate heatmaps, one per output class
    """
    output_dir = Path(output_path)
    output_dir.mkdir(parents=True, exist_ok=True)

    state_dict = torch.load(model_state_path, map_location="cpu")

    # Collect only 2D weight tensors (linear layers)
    layers = [
        (name, tensor)
        for name, tensor in state_dict.items()
        if isinstance(tensor, torch.Tensor) and tensor.ndim == 2
    ]

    if no_of_layers is not None:
        layers = layers[:no_of_layers]

    total_layers = len(layers)

    for layer_idx, (name, tensor) in enumerate(layers):
        weight = tensor.detach().cpu().float().numpy()  # Shape: (4, 60)
        
        print(f"\nLayer {layer_idx}: {name}")
        print(f"  Weight shape: {weight.shape}")
        
        num_output_classes = weight.shape[0]  # Should be 4
        n_features = weight.shape[1]          # Should be 60
        
        # Fixed structure for your case
        n_experts = 15
        n_input_classes = 4
        
        print(f"  Structure: {n_input_classes} input classes × {n_experts} experts = {n_features} features")
        print(f"  Output classes: {num_output_classes}")
        
        # Create separate heatmap for each output class
        for output_class_idx in range(num_output_classes):
            
            # Extract weights for this output unit (shape: 60,)
            class_weights = weight[output_class_idx, :]
            
            # Reshape to (n_input_classes, n_experts) then transpose to (n_experts, n_input_classes)
            # Input organized as: [C0_E0, C0_E1, ..., C0_E14, C1_E0, C1_E1, ..., C1_E14, ...]
            weight_matrix = class_weights.reshape(n_input_classes, n_experts).T  # Shape: (15, 4)
            
            # Create figure
            fig, ax = plt.subplots(figsize=(8, 10))
            
            # Heatmap with symmetric colormap centered at 0
            vmax = np.abs(weight_matrix).max()
            im = ax.imshow(weight_matrix, cmap='RdBu_r', aspect='auto',
                          vmin=-vmax, vmax=vmax)
            
            cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label('Weight Value', fontsize=12, fontweight='bold')
            
            # Labels
            ax.set_xlabel('Input Classes', fontsize=13, fontweight='bold')
            ax.set_ylabel('Pseudo-Experts', fontsize=13, fontweight='bold')
            ax.set_title(f'Weights for Output Class {output_class_idx}\n({n_experts} experts × {n_input_classes} input classes)',
                       fontsize=15, fontweight='bold', pad=15)
            
            # Set ticks
            ax.set_xticks(range(n_input_classes))
            ax.set_xticklabels([f'Class {i}' for i in range(n_input_classes)], fontsize=11)
            
            ax.set_yticks(range(n_experts))
            ax.set_yticklabels([f'Expert {i}' for i in range(n_experts)], fontsize=10)
            
            # Add grid between cells
            ax.set_xticks(np.arange(n_input_classes) - 0.5, minor=True)
            ax.set_yticks(np.arange(n_experts) - 0.5, minor=True)
            ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5, alpha=0.3)
            
            # Statistics box
            stats_text = (f'Mean: {weight_matrix.mean():.4f}\n'
                        f'Std: {weight_matrix.std():.4f}\n'
                        f'Min: {weight_matrix.min():.4f}\n'
                        f'Max: {weight_matrix.max():.4f}')
            
            ax.text(0.02, 0.98, stats_text,
                   transform=ax.transAxes,
                   fontsize=11,
                   verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.95, edgecolor='black'))
            
            plt.tight_layout()
            
            # Save
            safe_name = name.replace(".", "_").replace("/", "_")
            filename = output_dir / f"output_class_{output_class_idx}_{safe_name}.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()
            
            print(f"    ✓ Saved class {output_class_idx}: {filename.name}")
    
    print(f"\n✓ All visualizations saved to {output_dir}")

In [ ]:
for model_name in models:
    for normalization in normalizations:
        for statistic in statistics:
            for dataset in datasets:
                for interpolation in interpolations:
                    for layer in layers:
                        visualize_path = kfold_models_visualize_dir + f"/{model_name}/{dataset}/model_{interpolation}_{normalization}_simple_{no_of_layers}_{statistic}"
                        model_state_path = kfold_models_dir + f"/{model_name}/{dataset}/model_{interpolation}_{normalization}_simple_{no_of_layers}_{statistic}.pt"
                        visualize_weight_matrix(model_state_path,visualize_path,layer)
                                                


In [ ]:
# in the visualization 
# have input dim , output dim 
# for the last layer, get the no of classes , which is the no fo units in the output layer 
# for the vosualaization of the last layer , have the y axis which represents the utput dim , as class 0, class 1 ...class n 

## Visualisations to evaluate k fold performance of different configurations

In [ ]:
def generate_performance_heatmaps():
    """Generate 8 compact heatmaps with mean±std annotations."""
    
    datasets = ['ag_news', 'yelp_review']
    interpolations = ['model_baseline', 'slerp', 'ties', 'linear']
    normalizations = ['max', 'standard']
    layers = [2, 3, 4]
    statistics = ['mean', 'mean&std']
    models = ["bert","gpt2"]
    
    fig, axes = plt.subplots(4, 2, figsize=(12, 16))
    fig.suptitle('Validation MAE Performance', 
                 fontsize=20, fontweight='bold', y=0.995)
    
    plot_idx = 0
    
    for model_name in models:
        for dataset in datasets:
            for normalization in normalizations:
                for statistic in statistics:
                    # Build matrices for mean and std
                    mean_data = []
                    annot_data = []  # For displaying mean±std
                    
                    for layer in layers:
                        mean_row = []
                        annot_row = []
                        
                        for interpolation in interpolations:
                            try:
                                filename = kfold_results_dir + f"/{model_name}/{dataset}/results_{dataset}_{interpolation}_{normalization}_simple_{layer}_{statistic}.pkl"
                                with open(filename, 'rb') as f:
                                    dic = pl.load(f)[-1]
                                    mae = dic['avg_val_mae']
                                    std = dic['std_val_mae']
                                    
                                    mean_row.append(mae)
                                    annot_row.append(f'{mae:.3f}\n±{std:.3f}')
                            except:
                                mean_row.append(np.nan)
                                annot_row.append('N/A')
                        
                        mean_data.append(mean_row)
                        annot_data.append(annot_row)
                    
                    # DataFrame (for coloring)
                    df = pd.DataFrame(mean_data,
                                    index=[f'{l}L' for l in layers],
                                    columns=['Base', 'SLERP', 'TIES', 'Lin'])
                    
                    # Plot with custom annotations
                    ax = axes.flatten()[plot_idx]
                    sns.heatmap(df, 
                            annot=np.array(annot_data),  # ← Custom text
                            fmt='',  # ← Don't format, use raw strings
                            cmap='RdYlGn_r',
                            vmin=0.0, vmax=0.15, 
                            ax=ax, 
                            linewidths=1.5,
                            cbar_kws={'label': 'MAE'},
                            annot_kws={'fontsize': 10, 'fontweight': 'bold'},
                            cbar=True)
                    
                    # Larger axis labels
                    ax.set_title(f'{dataset.upper()}\n{normalization} | {statistic}',
                                fontsize=13, fontweight='bold', pad=8)
                    ax.set_xlabel('Method', fontsize=12, fontweight='bold')
                    ax.set_ylabel('Layers', fontsize=12, fontweight='bold')
                    
                    # Larger tick labels
                    ax.tick_params(axis='both', labelsize=11)
                    
                    plot_idx += 1
        
    plt.tight_layout(rect=[0, 0, 1, 0.99])
    plt.savefig('performance_heatmaps.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("✓ Saved: performance_heatmaps.png")

# Run
generate_performance_heatmaps()

In [ ]:
def generate_dataset_comparison_heatmaps():
    """Generate 4 figures comparing ag_news vs yelp_review side-by-side."""
    
    datasets = ['ag_news', 'yelp_review']
    interpolations = ['model_baseline', 'slerp', 'ties', 'linear']
    normalizations = ['max', 'standard']
    layers = [2, 3, 4]
    statistics = ['mean', 'mean&std']
    models = ["bert","gpt2"]
    
    for model_name in models:
        for normalization in normalizations:
            for statistic in statistics:
                
                fig, axes = plt.subplots(1, 2, figsize=(16, 6))
                fig.suptitle(f'Validation MAE Comparison: {normalization.upper()} | {statistic}',
                            fontsize=16, fontweight='bold', y=0.98)
                
                for dataset_idx, dataset in enumerate(datasets):
                    
                    mean_data = []
                    annot_data = []
                    
                    for layer in layers:
                        mean_row = []
                        annot_row = []
                        
                        for interpolation in interpolations:
                            filename = kfold_results_dir + f"/{model_name}/{dataset}/results_{dataset}_{interpolation}_{normalization}_simple_{layer}_{statistic}.pkl"

                            
                            with open(filename, 'rb') as f:
                                dic = pl.load(f)[-1]
                                mae = dic['avg_val_mae']
                                std = dic['std_val_mae']
                                
                                mean_row.append(mae)
                                annot_row.append(f'{mae:.3f}\n±{std:.3f}')
                        
                        mean_data.append(mean_row)
                        annot_data.append(annot_row)
                    
                    df = pd.DataFrame(mean_data,
                                    index=[f'{l}L' for l in layers],
                                    columns=['Base', 'SLERP', 'TIES', 'Lin'])
                    
                    ax = axes[dataset_idx]
                    sns.heatmap(df,
                            annot=np.array(annot_data),
                            fmt='',
                            cmap='RdYlGn_r',
                            vmin=0.0, vmax=0.15,
                            ax=ax,
                            linewidths=1.5,
                            annot_kws={'fontsize': 11, 'fontweight': 'bold'},
                            cbar=False)
                    
                    ax.set_title(f'{dataset.upper()}',
                                fontsize=14, fontweight='bold', pad=10)
                    
                    # Only show y-axis label on left subplot
                    if dataset_idx == 0:
                        ax.set_ylabel('Network Depth', fontsize=12, fontweight='bold')
                    else:
                        ax.set_ylabel('')
                    
                    # Only show x-axis label on bottom (both subplots)
                    ax.set_xlabel('')
                    
                    ax.tick_params(axis='both', labelsize=11)
                
                # Add shared x-axis label
                fig.text(0.5, 0.02, 'Interpolation Method', 
                        ha='center', fontsize=12, fontweight='bold')
                
                plt.tight_layout(rect=[0, 0.04, 1, 0.96])
                
                filename = f'comparison_{normalization}_{statistic}.png'
                plt.savefig(filename, dpi=300, bbox_inches='tight')
                plt.show()
                print(f"✓ Saved: {filename}")

generate_dataset_comparison_heatmaps()

In [ ]:
def plot_accuracy_histogram():
    """Plot accuracy distributions using histograms with KDE - LAST VALUES ONLY."""
    
    datasets = ['ag_news', 'yelp_review']
    models = ["bert","gpt2"]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Accuracy Distribution',
                fontsize=16, fontweight='bold', y=0.98)
    
    for model_name in models:
        for idx, dataset in enumerate(datasets):
            data_dir = datainfo_dir + f"/{model_name}/{dataset}"
            experiments = load_accuracy_data(dataset, data_dir)
            
            # Collect ONLY last accuracy value from each experiment
            all_acc_values = []
            for exp in experiments:
                acc_arr = exp['accuracy_arr']
                if isinstance(acc_arr, list):
                    acc_arr = np.array(acc_arr)
                
                # Take the last value (could be last element or last row's last element)
                last_val = acc_arr.flatten()[-1]  # ← Get last value
                all_acc_values.append(last_val)
            
            ax = axes[idx]
            
            # Histogram
            ax.hist(all_acc_values, bins=30, density=True, 
                    alpha=0.6, color='skyblue', edgecolor='black', linewidth=1.2,
                    range=(0, 1))
            
            # KDE
            from scipy.stats import gaussian_kde
            kde = gaussian_kde(all_acc_values)
            x_range = np.linspace(0, 1, 200)
            ax.plot(x_range, kde(x_range), 'r-', linewidth=2.5, label='KDE')
            
            ax.set_xlim(0, 1)
            ax.set_xlabel('Accuracy', fontsize=12, fontweight='bold')
            ax.set_ylabel('Density', fontsize=12, fontweight='bold')
            ax.set_title(f'{dataset.upper()}', fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')
            ax.legend(fontsize=10)
            
            # Stats
            mean_acc = np.mean(all_acc_values)
            median_acc = np.median(all_acc_values)
            std_acc = np.std(all_acc_values)
            
            stats_text = (f'Mean: {mean_acc:.4f}\nMedian: {median_acc:.4f}\n'
                        f'Std: {std_acc:.4f}\nn={len(all_acc_values)} configs')
            ax.text(0.98, 0.98, stats_text,
                transform=ax.transAxes,
                fontsize=10,
                verticalalignment='top',
                horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(f'accuracy_distribution_last_values_{model_name}.png', dpi=300, bbox_inches='tight')
        plt.show()
        
    print("\n✓ Saved: accuracy_distribution_last_values.png")

plot_accuracy_histogram()

In [ ]:
def distance_of_proportions(X, no_of_classes, ddof=0):
    """
    X: array shape (N,3) with values in [0,1]
    returns:
      P: normalized proportions, shape (N,3)
    """
    X = np.asarray(X, dtype=float)
    P = X / X.sum(axis=1, keepdims=True)
    l1 = np.sum(np.abs(P - float(1/no_of_classes)), axis=1)
    return l1

In [ ]:
def load_all_proportions(dataset, base_dir):
    """
    Load all proportion configurations for a dataset.
    
    Args:
        dataset: 'ag_news' or 'yelp_review'
        base_dir: Directory containing experiment folders
    
    Returns:
        List of proportion arrays
    """
    from pathlib import Path
    
    base_path = Path(base_dir)
    
    # Find all directories for this dataset
    all_dirs = [d for d in base_path.iterdir() 
                if d.is_dir() and d.name.startswith(dataset)]
    
    proportions_list = []
    
    for exp_dir in all_dirs:
        props = parse_proportion_from_filename(exp_dir.name)
        if props is not None:
            proportions_list.append(props)
    
    print(f"✓ Found {len(proportions_list)} configurations for {dataset}")
    return proportions_list

# Usage
if __name__ == "__main__":
    
    datasets = ['ag_news', 'yelp_review']
    models = ["bert","gpt2"]
    
    for model_name in models:
        for dataset in datasets:
            base_dir = datainfo_dir + f"/{model_name}/{dataset}"
            proportions = load_all_proportions(dataset, base_dir)
            
            print(f"\n{dataset.upper()} Proportions:")
            for p in proportions[:5]:  # Show first 5
                print(f"  {p}")

In [ ]:
def plot_proportion_distance_distribution():
    """Plot distribution of proportion distances from uniform."""
    
    datasets = ['ag_news', 'yelp_review']
    models = ["bert","gpt2"]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Distribution of mean Proportion Distances from Uniform',
                fontsize=16, fontweight='bold', y=0.98)
    
    for model_name in models:
        for idx, dataset in enumerate(datasets):
            
            if dataset == 'ag_news':            
                base_dir = datainfo_dir + f"/{model_name}/{dataset}"
                n_classes = 4
                
            else:  # yelp_review
                base_dir = datainfo_dir + f"/{model_name}/{dataset}"
                n_classes = 5
            
            # Load all proportions using existing function
            proportions_list = load_all_proportions(dataset, base_dir)
            
            # Compute distances
            distances = [compute_proportion_distance(props, n_classes) 
                        for props in proportions_list]
            
            ax = axes[idx]
            
            # Histogram with fixed range
            ax.hist(distances, bins=30, density=True, 
                    alpha=0.6, color='skyblue', edgecolor='black', 
                    linewidth=1.2, range=(0, 1))  # ✅ Fixed range
            
            # KDE overlay (computed on full data, plotted over [0,1])
            if len(distances) > 1:
                from scipy.stats import gaussian_kde
                kde = gaussian_kde(distances)
                x_range = np.linspace(0, 1, 200)  # ✅ Fixed x-range
                ax.plot(x_range, kde(x_range), 'r-', 
                    linewidth=2.5, label='KDE')
            
            # Statistics
            mean_dist = np.mean(distances)
            median_dist = np.median(distances)
            std_dist = np.std(distances)
            min_dist = np.min(distances)
            max_dist = np.max(distances)
            
            # Formatting
            ax.set_xlim(0, 1)  # ✅ Force x-axis limits
            ax.set_xlabel('Average L1 Distance from Uniform', 
                        fontsize=12, fontweight='bold')
            ax.set_ylabel('Density', fontsize=12, fontweight='bold')
            ax.set_title(f'{dataset.upper()} ({n_classes} classes)', 
                        fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')
            ax.legend(fontsize=10)
            
            # Add statistics box
            stats_text = (f'Mean: {mean_dist:.4f}\n'
                        f'Median: {median_dist:.4f}\n'
                        f'Std: {std_dist:.4f}\n'
                        f'Min: {min_dist:.4f}\n'
                        f'Max: {max_dist:.4f}\n'
                        f'n={len(distances)}')
            
            ax.text(0.98, 0.98, stats_text,
                transform=ax.transAxes,
                fontsize=10,
                verticalalignment='top',
                horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(f'proportion_distance_distribution_{model_name}.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print("\n✓ Saved: proportion_distance_distribution.png")


def compute_proportion_distance(proportions, n_classes):
    """Compute L1 distance from uniform distribution."""
    proportions = np.array(proportions)
    uniform = np.full(n_classes, 1.0 / n_classes)
    return np.mean(np.abs(proportions - uniform))


# Run it
plot_proportion_distance_distribution()